In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install sentence-transformers umap-learn hdbscan matplotlib -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap
import hdbscan
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("LaBSE")

# ── Seed vocabulary: common words across many languages ──────────────────────
# In production: pull from Wiktionary frequency lists or Universal Dependencies
# This is the bootstrapped version with a curated starter set
MULTILINGUAL_VOCAB = {
    "en": ["water","fire","earth","sky","love","fear","trust","light","dark","time",
           "life","death","eat","sleep","run","give","take","speak","think","feel"],
    "es": ["agua","fuego","tierra","cielo","amor","miedo","confianza","luz","oscuridad","tiempo",
           "vida","muerte","comer","dormir","correr","dar","tomar","hablar","pensar","sentir"],
    "de": ["Wasser","Feuer","Erde","Himmel","Liebe","Angst","Vertrauen","Licht","Dunkel","Zeit",
           "Leben","Tod","essen","schlafen","laufen","geben","nehmen","sprechen","denken","fühlen"],
    "fr": ["eau","feu","terre","ciel","amour","peur","confiance","lumière","obscurité","temps",
           "vie","mort","manger","dormir","courir","donner","prendre","parler","penser","ressentir"],
    "pt": ["água","fogo","terra","céu","amor","medo","confiança","luz","escuridão","tempo",
           "vida","morte","comer","dormir","correr","dar","tomar","falar","pensar","sentir"],
    "ja": ["水","火","土","空","愛","恐怖","信頼","光","闇","時間",
           "生命","死","食べる","寝る","走る","与える","取る","話す","考える","感じる"],
}

CONCEPT_LABELS = ["water","fire","earth","sky","love","fear","trust","light",
                  "dark","time","life","death","eat","sleep","run","give",
                  "take","speak","think","feel"]

SPEAKER_WEIGHTS = {"en":1500,"es":560,"de":130,"fr":280,"pt":260,"ja":125}

# ── Embed all words ───────────────────────────────────────────────────────────
records = []
for lang, words in MULTILINGUAL_VOCAB.items():
    for concept, word in zip(CONCEPT_LABELS, words):
        records.append({"lang": lang, "concept": concept, "word": word})

df = pd.DataFrame(records)
embs = model.encode(df["word"].tolist(), normalize_embeddings=True, show_progress_bar=True)
df["embedding"] = list(embs)

# ── Compute universal concept centroids (speaker-weighted) ────────────────────
centroids = {}
for concept in CONCEPT_LABELS:
    sub = df[df["concept"] == concept]
    w   = np.array([SPEAKER_WEIGHTS[l] for l in sub["lang"]], dtype=float)
    w  /= w.sum()
    c   = np.stack(sub["embedding"].values)
    centroid = (c * w[:, None]).sum(axis=0)
    centroid /= np.linalg.norm(centroid)
    centroids[concept] = centroid

centroid_matrix = np.stack([centroids[c] for c in CONCEPT_LABELS])

# ── UMAP projection ───────────────────────────────────────────────────────────
reducer = umap.UMAP(n_components=2, random_state=42, metric="cosine", n_neighbors=8)
all_embs = np.vstack([embs, centroid_matrix])
projected = reducer.fit_transform(all_embs)

word_proj     = projected[:len(embs)]
centroid_proj = projected[len(embs):]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 9))
colors = {"en":"#E24B4A","es":"#378ADD","de":"#1D9E75","fr":"#EF9F27","pt":"#D4537E","ja":"#7F77DD"}

for lang in MULTILINGUAL_VOCAB:
    mask = df["lang"] == lang
    ax.scatter(word_proj[mask, 0], word_proj[mask, 1],
               c=colors[lang], s=40, alpha=0.7, label=lang, zorder=2)

# Centroids as stars
ax.scatter(centroid_proj[:, 0], centroid_proj[:, 1],
           c="black", s=150, marker="*", zorder=4, label="universal centroid")
for i, concept in enumerate(CONCEPT_LABELS):
    ax.annotate(concept, (centroid_proj[i,0]+0.02, centroid_proj[i,1]+0.02),
                fontsize=8, fontweight="bold")

ax.set_title("Universal concept centroids in multilingual embedding space\n"
             "(stars = speaker-weighted centroids = proto-tokens for universal language)")
ax.legend(title="Language", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.axis("off")
plt.tight_layout()
plt.savefig("universal_language_bootstrap.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nNext steps to extend this into a full bootstrapper:")
print("1. Load a 50k-word frequency list per language (Wiktionary dumps)")
print("2. For each centroid, find nearest word in each language = proto-definition")
print("3. Assign phonetic token: most common sound pattern across the cluster")
print("4. Build grammar rules by analysing which grammatical markers cluster tightly")